#### Deterministic-learning Algorithm to find the optimal policy

This algorithm, like Q-learning, learns the quality of eeach state under each action.

For each state and action, it averages the profit one time unit after under all possible options multiplied by the probability of their happening:
* Since demand is a Poisson distribution, it checks every possible demand multiplied for the probability of it happening (truncated by the tolerance `tol' on its probability)
* Since the damaged aquariums follow a binomial distribution, it runs through every possible case of damage (0 to all aquariums in stock) multiplied by the probability of it happening


Because we don't have to run through $10^6$ weeks, this algorithm is much faster.
  


In [167]:
import numpy as np
import math
import random as rdm


Lambda=1; m=3;
d=5; s=20;
rho=0.01; c=60;

# nweeks = 1000000

tol=0.00000000001
N=1000
Dprob=[]
for k in range(N):
    if (Lambda**k*np.exp(-Lambda)/math.factorial(k)<tol):
        break
    Dprob.append(Lambda**k*np.exp(-Lambda)/math.factorial(k))
Nd = k


def Qlearning(m, rho, Dprob):

    alpha=1
    a_factor = (0.001/alpha)**(1/nweeks)     
        # a_factor has to be really close to 1, otherwise after a few weeks, 
        # the learning stops because alpha becomes too small.
        # This formula ensures that at the end of the simulation, alpha is 0.001
    gamma = 0.85

    # K has binomial probability
    Kprob=np.zeros([m+1,m+1])
    for x in range(m+1):
        for k in range(0,x+1):
            Kprob[x,k] = math.factorial(x)/(math.factorial(k)*math.factorial(x-k))*(rho**k)*(1-rho)**(x-k)

    
    Q = np.zeros([m+1,m+1])
    

    for xn in range(m+1):
        for a in range(0,m-xn+1):
            xnp1 = xn + a
            

            R=0            
            for dnp1 in range(Nd):
                for kn in range(0,xnp1+1):
                    DamageCost = kn*c

                    # xnp2 = xnp1 - np.min([dnp1,xnp1])
                    R += (s*np.min([dnp1,xnp1-rho*xnp1]) - DamageCost - d*(dnp1*(xnp1-rho*xnp1)>0))*Dprob[dnp1]*Kprob[xnp1,kn]
            Q[xn, a] = R
    p=[]
    for i in range(m+1):
        if np.max(Q[i,:])>0:
            p.append(np.argmax(Q[i,:]))
        else:
            p.append(0)

    print('m =',m,' rho =',rho,' -> ',p)
    
    # return p
    return Q




In [172]:
Q=Qlearning(5, 0.005, Dprob)
print(Nd)
print(Q)

m = 5  rho = 0.005  ->  [4, 3, 2, 1, 0, 0]
14
[[ 0.          9.11859633 14.11378251 15.44856826 15.54482655 15.32378883]
 [ 9.11859633 14.11378251 15.44856826 15.54482655 15.32378883  0.        ]
 [14.11378251 15.44856826 15.54482655 15.32378883  0.          0.        ]
 [15.44856826 15.54482655 15.32378883  0.          0.          0.        ]
 [15.54482655 15.32378883  0.          0.          0.          0.        ]
 [15.32378883  0.          0.          0.          0.          0.        ]]


In [173]:
Q=Qlearning(3, 0.005, Dprob)
print(Nd)
print(Q)

m = 3  rho = 0.005  ->  [3, 2, 1, 0]
14
[[ 0.          9.11859633 14.11378251 15.44856826]
 [ 9.11859633 14.11378251 15.44856826  0.        ]
 [14.11378251 15.44856826  0.          0.        ]
 [15.44856826  0.          0.          0.        ]]


#### Observations

Observe that when we learn the best outcome with this new Q-learn method, the matrix $Q$ is always symmetric, which means that the solution is always going to by of the form:
* $n \quad (n-1) \quad \cdots \quad 0 \cdots 0$

This is because if one state has the best quality, then the best policy is to be on that state as mush as possible, which means re-stock whenever we are below that state and not re-stock if we start above.


This means that the uncertainty of the damages penalizes a lot more than the certainty of having the mean averages happening consistently. 